# Trabalho 1 - Recuperação da Informação

Grupo:

- Arthur Trottmann Ramos (14681052)
- Maicon Chaves Marques (14593530)

## Instalação de Dependências e Carregamento de Dataset

In [3]:
pip install NLTK numpy ir_datasets

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 17.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------  12.3/12.6 MB 61.5 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 50.0 MB/s  0:00:00
   ---------------------------------------- 0.0/947.8 kB ? eta -:--:--
   ---------------------------------------- 947.8/947.8 kB 31.7 MB/s  0:00:00
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 4.1/4.1 MB 35.7 MB/s  0:00:00

   ----------------------------------------  0/16 [urllib3]
   -- -------------------------------------  1/16 [tqdm]
   ------- --------------------------------  3/16 [pyyaml]
   ---------- -----------------------------  4/16 [numpy]
   ---------- -----------------------------  4/16 [numpy]
   ---------- -----------------------------  4/

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
import ir_datasets

dataset = ir_datasets.load("cranfield")

## Pré-Processamento

In [6]:
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stemmer = PorterStemmer()


def tokenization(text):
  tokenizer = RegexpTokenizer(r'\w+')
  clean_tokens = tokenizer.tokenize(text)
  return lower_case_normalization(clean_tokens)

def remove_stopwords(words):
  stopwords_set = set(stopwords.words('english'))
  filtered_words = [word for word in words if word not in stopwords_set]
  return filtered_words

def lower_case_normalization(words):
  normalized_words = [word.lower() for word in words]
  return normalized_words

def stemming(words):
  stemmed_words = [stemmer.stem(word) for word in words]
  return stemmed_words

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Arthur\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [7]:
def preprocess(text, config_type=0):
  words = tokenization(text)

  if config_type == 1:
    words = remove_stopwords(words)
  if config_type == 2:
    words = stemming(words)
  if config_type == 3:
    words = remove_stopwords(words)
    words = stemming(words)

  return words

## Índice Invertido

In [8]:
class InvertedIndex:
  def __init__(self):
    self.index = {}
    self.document_length = {}
    self.n_documents = 0
    self.avgdl = 0.0

  def build(self, documents, preprocessing_type):
    for doc in documents:
      doc_id = doc[0]
      doc_title = doc[1]
      doc_text = doc[2]
      doc_author = doc[3]

      self.n_documents += 1

      title_words = preprocess(doc_title, preprocessing_type)
      text_words = preprocess(doc_text, preprocessing_type)
      author_words = preprocess(doc_author, preprocessing_type)

      self.document_length[doc_id] = len(title_words) + len(text_words) + len(author_words)

      for word in title_words:
        if word not in self.index:
          self.index[word] = {}
        if doc_id not in self.index[word]:
          self.index[word][doc_id] = 0
        self.index[word][doc_id] += 1

    self.avgdl = sum(self.document_length.values()) / len(self.document_length)

## Modelo Probabilístico (BM25)

## Modelo Vetorial